# UCS420: Cognitive Computing — Assignment 4: A Cognitive FAQ System Using Pandas (Nova 2.0)

**Set your roll number in the next cell** — the last two digits decide the categories of your two personalised entries.

In [1]:
import pandas as pd
import re

ROLL_NUMBER = "102203045"        # <-- replace with YOUR roll number (digits only)

## Q1. Personalised knowledge base (6 rows)

In [2]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

CATEGORIES = ["billing", "account", "general"]

# pool of invented entries per category (the digit picks the category, the position picks the template)
templates = {
    "billing": [
        {"question": "how do i get a refund", "answer": "Raise a refund request from Billing > Refunds; it is processed in 5-7 working days.",
         "keywords": "refund money return"},
        {"question": "where can i download my invoice", "answer": "Open Billing > Invoices and click Download PDF.",
         "keywords": "invoice receipt download"},
    ],
    "account": [
        {"question": "how do i update my registered mobile number", "answer": "Go to Profile > Contact Details and verify the new number with an OTP.",
         "keywords": "mobile number update"},
        {"question": "how do i delete my account", "answer": "Open Settings > Privacy > Delete Account and confirm with your password.",
         "keywords": "delete close account"},
    ],
    "general": [
        {"question": "where is the campus office located", "answer": "The office is on the ground floor of the main building.",
         "keywords": "office location address"},
        {"question": "how can i contact support", "answer": "Email support@example.com or call the helpline during working hours.",
         "keywords": "contact support helpline"},
    ],
}

last_two = [int(d) for d in re.sub(r"\D", "", ROLL_NUMBER)[-2:]]
personal = []
for position, d in enumerate(last_two):
    cat = CATEGORIES[d % 3]
    # use a different template if both digits map to the same category
    used_before = sum(1 for p in personal if p["category"] == cat)
    entry = dict(templates[cat][used_before % len(templates[cat])])
    entry["category"] = cat
    personal.append(entry)
    print(f"digit {d} -> category[{d} % 3] = {cat}")

faq = pd.DataFrame(fixed_entries + personal)[["question", "answer", "keywords", "category"]]
pd.set_option("display.max_colwidth", None, "display.width", 200)
print()
print(faq)

digit 4 -> category[4 % 3] = account
digit 5 -> category[5 % 3] = general

                                      question                                                                  answer                 keywords category
0                       what is the annual fee                                               The annual fee is Rs 500.    fee cost price charge  billing
1                        how to reset password                                        Go to Settings > Reset Password.     password reset login  account
2                  what are your working hours                                               We are open 9 AM to 5 PM.   hours timing open time  general
3                        how can i pay the fee                              You can pay via UPI, card, or net banking.      pay payment upi fee  billing
4  how do i update my registered mobile number  Go to Profile > Contact Details and verify the new number with an OTP.     mobile number update  account
5      

## Q2. Scoring function

*Confidence* of an entry for a query:

1. Remove common filler words from the query (`what`, `is`, `the`, `how`, …) so they cannot create matches.
2. Each remaining query term scores **2** if it is one of the entry's *keywords*, **1** if it only appears in the
   entry's *question*, and **0** otherwise.
3. `confidence = total / (2 × number_of_query_terms)`, so it lies between 0 and 1 (1 = every term is a keyword).

Only entries with confidence > 0 are returned, ranked from best to worst.

In [3]:
STOP = {"what", "is", "the", "how", "to", "can", "i", "do", "does", "are", "your", "a", "an", "of", "my",
        "you", "for", "in", "on", "where", "me", "we", "it", "and", "or"}

def query_terms(query):
    return [t for t in re.findall(r"[a-z0-9]+", query.lower()) if t not in STOP]

def score_query(query, df):
    terms = query_terms(query)
    if not terms:
        return df.iloc[0:0].assign(confidence=[])
    scores = []
    for _, row in df.iterrows():
        kw = set(row["keywords"].lower().split())
        qw = set(re.findall(r"[a-z0-9]+", row["question"].lower()))
        pts = sum(2 if t in kw else 1 if t in qw else 0 for t in terms)
        scores.append(pts / (2 * len(terms)))
    out = df.assign(confidence=scores)
    out = out[out["confidence"] > 0].sort_values("confidence", ascending=False, kind="stable")
    return out

print(score_query("how can I reset my password", faq)[["question", "confidence"]])
print()
print(score_query("annual fee", faq)[["question", "confidence"]])

                question  confidence
1  how to reset password         1.0

                 question  confidence
0  what is the annual fee        0.75
3   how can i pay the fee        0.50


## Q3. `same_category`

In [4]:
def same_category(category_name, df):
    """Return all questions that belong to the given category."""
    return df.loc[df["category"] == category_name, "question"].tolist()

cat_of_personal = faq.iloc[4]["category"]           # category of the first personalised entry
print(f"Questions in category '{cat_of_personal}':")
for q in same_category(cat_of_personal, faq):
    print(" -", q)

Questions in category 'account':
 - how to reset password
 - how do i update my registered mobile number


## Q4. Add a keyword (user input) and save to CSV

In [5]:
entry_idx = 0                                        # entry 0: "what is the annual fee"
try:
    new_kw = input(f"Enter a new keyword for '{faq.loc[entry_idx, 'question']}': ").strip()
except Exception:                                    # non-interactive run (e.g. notebook executed headlessly)
    new_kw = "annual"
    print("(no interactive input available — using demo keyword 'annual')")

if new_kw and new_kw.lower() not in faq.loc[entry_idx, "keywords"].split():
    faq.loc[entry_idx, "keywords"] += " " + new_kw.lower()

csv_name = f"{ROLL_NUMBER}_faq_data.csv"
faq.to_csv(csv_name, index=False)
print("updated keywords:", faq.loc[entry_idx, "keywords"])
print("saved ->", csv_name)
print(pd.read_csv(csv_name))

(no interactive input available — using demo keyword 'annual')
updated keywords: fee cost price charge annual
saved -> 102203045_faq_data.csv
                                      question                                                                  answer                      keywords category
0                       what is the annual fee                                               The annual fee is Rs 500.  fee cost price charge annual  billing
1                        how to reset password                                        Go to Settings > Reset Password.          password reset login  account
2                  what are your working hours                                               We are open 9 AM to 5 PM.        hours timing open time  general
3                        how can i pay the fee                              You can pay via UPI, card, or net banking.           pay payment upi fee  billing
4  how do i update my registered mobile number  Go to Profile > Cont

## Q5. Entries per category (groupby)

In [6]:
print(faq.groupby("category").size().rename("entries"))

category
account    2
billing    2
general    2
Name: entries, dtype: int64


## Q6. Tie-aware answering
If several entries share the highest confidence, **all** of them are printed instead of silently picking one.

In [7]:
def answer_query(query, df):
    ranked = score_query(query, df)
    print(f'Query: "{query}"')
    if ranked.empty:
        print("  No matching entry found.\n")
        return ranked
    best = ranked["confidence"].max()
    top = ranked[ranked["confidence"] == best]
    if len(top) > 1:
        print(f"  TIE: {len(top)} entries share the highest confidence ({best:.2f}) — showing all of them:")
    else:
        print(f"  Best match (confidence {best:.2f}):")
    for _, r in top.iterrows():
        print(f"   * Q: {r['question']}\n     A: {r['answer']}")
    print()
    return top

_ = answer_query("fee", faq)                 # tie: both fee entries
_ = answer_query("reset my password", faq)   # unique best match
_ = answer_query("annual fee", faq)          # unique best match (only one entry contains "annual")

Query: "fee"
  TIE: 2 entries share the highest confidence (1.00) — showing all of them:
   * Q: what is the annual fee
     A: The annual fee is Rs 500.
   * Q: how can i pay the fee
     A: You can pay via UPI, card, or net banking.

Query: "reset my password"
  Best match (confidence 1.00):
   * Q: how to reset password
     A: Go to Settings > Reset Password.

Query: "annual fee"
  Best match (confidence 1.00):
   * Q: what is the annual fee
     A: The annual fee is Rs 500.

